In [73]:
import ipywidgets as widgets
from IPython.display import display

# input creation using ipywidgets
fn_in = widgets.Text(description='First Name:')
ln_in = widgets.Text(description='Last Name:')
em_in = widgets.Text(description='Email:')
co_in = widgets.Text(description='Courses:', placeholder='Separated by commas')
li_in = widgets.Text(description='Link:')
btn = widgets.Button(description='Add to JSON', button_style='success')

def on_click(b):
    print(f"Ready to save {fn_in.value}!")

btn.on_click(on_click)
display(widgets.VBox([fn_in, ln_in, em_in, co_in, li_in, btn]))

In [ ]:
import json
from google.colab import drive

drive.mount('/content/drive')
student_data = {
    "first_name": name,
    "last_name": last_name,
    "email": email,
    "courses_this_semester": courses_this_semester,
    "interesting_link": interesting_link
    }
file_path = '/content/drive/My Drive/students.json'

if os.path.exists(file_path):
    with open(file_path, "r") as fid:
        try:
            data = json.load(fid)

            if not isinstance(data, list):
                data = [data]
        except json.JSONDecodeError:
            data = []
else:
    data = []


user_exists = any(student.get('email') == student_data['email'] for student in data)

if user_exists:
    print(f"Bummer! A student with the email '{student_data['email']}' is already in the system.")
else:
    # 4. Only append if they don't exist
    data.append(student_data)

with open(file_path, "w") as fid:
    json.dump(data, fid, indent=4)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

class StudentDirectoryApp:
    def __init__(self, file_path):
        self.file_path = file_path
        self.student_details_map = {}
        self.display_names = []

        # Ensure Drive is mounted if running in Colab
        if 'google.colab' in str(get_ipython()):
            if not os.path.exists('/content/drive/MyDrive'):
                from google.colab import drive
                drive.mount('/content/drive')

        self.load_student_data()
        self.build_ui()

    def load_student_data(self):
        """Reads and parses the JSON file."""
        self.student_details_map.clear()
        self.display_names.clear()

        if os.path.exists(self.file_path):
            try:
                with open(self.file_path, 'r') as f:
                    data = json.load(f)

                    # If your JSON is a list of students, we loop through them
                    # If it's a single student dictionary, we wrap it in a list
                    students = data if isinstance(data, list) else [data]

                    for s in students:
                        full_name = f"{s.get('first_name', 'Unknown')} {s.get('last_name', 'Student')}"
                        self.student_details_map[full_name] = s
                        self.display_names.append(full_name)
            except Exception as e:
                print(f"Error loading JSON: {e}")

    def update_display(self, change):
        """Updates the info card using dictionary keys from the JSON."""
        selected = change.get('new')
        details = self.student_details_map.get(selected)

        with self.info_output:
            clear_output(wait=True)
            if details:
                # Extracting data using JSON keys
                email = details.get('email', 'N/A')
                # Joining courses list into a string
                courses = ", ".join(details.get('courses_this_semester', []))
                link = details.get('interesting_link', '#')

                html = f"""
                <div class='info-card'>
                    <div style='margin-bottom:8px;'><strong>Name:</strong> {selected}</div>
                    <div style='margin-bottom:8px;'><strong>Email:</strong> {email}</div>
                    <div style='margin-bottom:8px;'><strong>Courses:</strong> {courses}</div>
                    <div style='margin-bottom:8px;'><strong>Link:</strong> <a href="{link}" target="_blank">{link}</a></div>
                </div>
                """
                display(widgets.HTML(html))

    def build_ui(self):
        """Constructs and styles the widgets."""
        if not self.display_names:
            display(widgets.HTML("<div style='color: red;'>Error: No valid student data found.</div>"))
            return

        self.style_html = widgets.HTML("""
        <style>
            .info-card {
                background-color: #ffffff;
                border-radius: 8px;
                padding: 20px;
                border-left: 5px solid #007bff;
                margin-top: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.05);
            }
        </style>
        <h2>Student Directory</h2>
        """)

        self.dropdown = widgets.Dropdown(
            options=self.display_names,
            description='Select Student:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='100%')
        )

        self.fav_show_input = widgets.Text(
            placeholder='e.g., Breaking Bad',
            description='Fav Show:',
            layout=widgets.Layout(width='65%')
        )

        self.save_button = widgets.Button(
            description='Save',
            button_style='success',
            icon='save',
            layout=widgets.Layout(width='30%')
        )

        self.info_output = widgets.Output()
        self.status_output = widgets.Output()

        self.dropdown.observe(self.update_display, names='value')
        self.save_button.on_click(self.save_favorite_show)

        input_row = widgets.HBox([self.fav_show_input, self.save_button],
                                layout=widgets.Layout(justify_content='space-between', width='100%'))

        main_container = widgets.VBox([
            self.style_html, self.dropdown, self.info_output, input_row, self.status_output
        ], layout=widgets.Layout(width='600px', padding='30px', border='1px solid #ddd', margin='20px 0'))

        self.ui = widgets.HBox([main_container], layout=widgets.Layout(justify_content='center', width='100%'))
        display(self.ui)
        self.update_display({'new': self.dropdown.value})

    def save_favorite_show(self, b):
        """Saves the show preference to a separate text log."""
        selected_student = self.dropdown.value
        show_name = self.fav_show_input.value.strip()

        if show_name and selected_student:
            with self.status_output:
                clear_output(wait=True)
                try:
                    # Log the favorite show to a text file
                    with open('/content/drive/My Drive/student_favorites.txt', 'a') as f:
                        f.write(f"{selected_student}: {show_name}\n")
                    display(widgets.HTML(f"<div style='color: green;'>Success: Saved '{show_name}'!</div>"))
                    self.fav_show_input.value = ''
                except Exception as e:
                    display(widgets.HTML(f"<div style='color: red;'>Error: {e}</div>"))

# --- Execution ---
JSON_FILE_PATH = '/content/drive/My Drive/students.json'
app = StudentDirectoryApp(JSON_FILE_PATH)

In [ ]:
import json
import os

def remove_student_by_email(email_to_remove, file_path):
    if not os.path.exists(file_path):
        print("Error: File not found.")
        return

    with open(file_path, "r") as fid:
        try:
            data = json.load(fid)
            if not isinstance(data, list):
                data = [data]
        except json.JSONDecodeError:
            print("Error: Could not read JSON data.")
            return


    starting_count = len(data)
    updated_data = [student for student in data if student.get('email') != email_to_remove]

    if len(updated_data) < starting_count:
        with open(file_path, "w") as fid:
            json.dump(updated_data, fid, indent=4)
        print(f"Success: Student with email '{email_to_remove}' has been removed.")
    else:
        print(f"No student found with email '{email_to_remove}'. No changes made.")

FILE_PATH = '/content/drive/My Drive/students.json'
# remove_student_by_email("nikosav@gmail.com", FILE_PATH)